# Exercise: Multi-Language Text Processing
## AIAT 121 - Natural Language Processing

## Learning Objectives

- Implement tokenization for three writing systems: Latin, Arabic and Chinese
- Compare tokenization methods on the same real text and explain where they disagree
- Compute per-language statistics on a **parallel** document — the same content in three languages
- Tokenize messy operational text from a real 911 call log and decide what a "token" should be there

## 🔗 Where this fits

**Builds on:** Unit 1 (the preprocessing pipeline) and Unit 2 example `01_advanced_tokenization.ipynb`, which
measured how many subword pieces English and Arabic cost per word. **Used later in:** Unit 3, where TF-IDF
is built over tokens, and Unit 4 lesson 03, where the same Arabic-costs-more effect shows up inside BERT's
own tokenizer.

## Real-World Context

You are building a text-analysis system for an organisation that publishes the same document in Arabic,
English and Chinese, and that also processes operational logs written in English by people in a hurry.
Two real sources are provided by the data cell — nothing to type, nothing to download:

- **The Universal Declaration of Human Rights**, Articles 1-28, in all three languages. The UN General
  Assembly adopted it on 10 December 1948 and it has been translated into more than 500 languages; the same
  content in three scripts makes it a *parallel* corpus, which is what makes a cross-language comparison
  meaningful. (Why 1-28 and not 30: the English copy in NLTK's corpus is cut off in the middle of Article
  29, so all three were trimmed to the articles they share. Real corpora arrive truncated.)
- **Real 911 call descriptions** from the Montgomery County, PA emergency log, through the shared loader:
  ALL CAPS, semicolons, `&`, `@`, timestamps, station codes.

**Task**: tokenize all of it correctly, measure what each choice costs, and write down what a token *is* in
each source.

---


## 📥 Inputs & 📤 Outputs

**Inputs:** three UTF-8 text files in `udhr/` next to this notebook (`udhr_en.txt`, `udhr_ar.txt`,
`udhr_zh.txt`, about 30 KB in total; fetched from the course repository only when there is no local clone),
and the 25,000-row 911 sample via `tools.data.load`. NLTK is optional — the regex module is enough for
every task.

**Outputs:** whatever your completed TODO cells print — the three tokenizers' results on Article 1, the
tokenization-method comparison, per-language statistics on the parallel articles, and your analysis of the
911 descriptions. The TODO cells produce no output until you complete them.

---


## The Dataset

Run this cell first. Every task below works on `articles` (a dict: language code → list of 28 article
strings, in the same order in every language) and on `calls` (a DataFrame of 911 records).


In [1]:
# --- Data setup. Works from any folder, and on Google Colab. -------------------------
# WHAT: find the repository root, put it on sys.path, then import the shared data loader.
# WHY:  a hard-coded relative path only resolves when the kernel's working directory happens
#       to be this notebook's folder. This does not care where you opened it from.
import sys, pathlib

_here = pathlib.Path.cwd().resolve()
_root = next((p for p in [_here, *_here.parents] if (p / "tools" / "data.py").exists()), None)
if _root is None:                     # Google Colab, or a stray copy of the notebook
    import urllib.request
    pathlib.Path("tools").mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/A-Alwabel/"
            "AI-Diploma-Program/main/tools/data.py", "tools/data.py")
    except Exception as _e:
        raise RuntimeError(
            "Could not find the AI Diploma repository from this folder, and could not "
            "download the data loader either. Open this notebook inside a clone of "
            "https://github.com/A-Alwabel/AI-Diploma-Program, or connect to the internet "
            f"and re-run this cell. (underlying error: {_e})") from None
    _root = pathlib.Path.cwd()
sys.path.insert(0, str(_root))

from tools.data import load
# -------------------------------------------------------------------------------------

import re

# The three UDHR files ship in the repository next to this notebook. They are the NLTK
# `udhr` corpus copies of the UN text, byte-for-byte except for two documented edits: all
# three are cut after Article 28 (the English copy stops mid-Article 29), and the Chinese
# copy had a space between every character — an artefact of the file, not of Chinese —
# which was removed so the text reads as Chinese is actually written.
DATA_REL = "Course 07/unit2-tokenization-morphology/exercises/udhr"
data_dir = _root / DATA_REL
if not (data_dir / "udhr_en.txt").exists():          # no clone: fetch the three small files once
    import urllib.parse, urllib.request
    data_dir.mkdir(parents=True, exist_ok=True)
    for f in ("udhr_en.txt", "udhr_ar.txt", "udhr_zh.txt"):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/A-Alwabel/AI-Diploma-Program/main/"
                                   + urllib.parse.quote(f"{DATA_REL}/{f}"), data_dir / f)

udhr = {lang: (data_dir / f"udhr_{lang}.txt").read_text(encoding="utf-8") for lang in ("en", "ar", "zh")}

# Even "find Article 5" is script-specific: the marker is a Latin word plus digits, an
# Arabic word plus digits, or Chinese numerals between two characters.
ARTICLE_MARK = {"en": r"Article \d+\s*\n", "ar": r"المادة\s*\d+\s*\n", "zh": r"第[一二三四五六七八九十]+条\s*\n"}
articles = {lang: [a.strip() for a in re.split(ARTICLE_MARK[lang], text)[1:]] for lang, text in udhr.items()}

for lang in ("en", "ar", "zh"):
    print(f"{lang}: {len(udhr[lang]):,} characters, {len(articles[lang])} articles")
assert len({len(a) for a in articles.values()}) == 1, "the three languages must have the same number of articles"
print("\nArticle 1, three scripts, same content:")
for lang in ("en", "ar", "zh"):
    print(f"  [{lang}] {articles[lang][0]}")

# The second source: real emergency-call descriptions, exactly as the dispatch system logged them.
calls = load("montgomery_911_calls", prefer="sample")   # prefer="sample": the same 25,000 rows for everyone
descriptions = calls["desc"].astype(str).tolist()
print(f"\n911 records: {len(calls):,}; columns: {list(calls.columns)}")
print("Three descriptions, as logged:")
for d in descriptions[:3]:
    print(f"  {d!r}")
print("Three titles:", calls["title"].head(3).tolist())


en: 9,932 characters, 28 articles
ar: 7,094 characters, 28 articles
zh: 2,816 characters, 28 articles

Article 1, three scripts, same content:
  [en] All human beings are born free and equal in dignity and rights. They are endowed with reason and conscience and should act towards one another in a spirit of brotherhood.
  [ar] يولد جميع الناس أحرارًا متساوين في الكرامة والحقوق. وقد وهبوا عقلاً وضميرًا وعليهم أن يعامل بعضهم بعضًا بروح الإخاء.
  [zh] 人人生而自由,在尊严和权利上一律平等。他们赋有理性和良心,并应以兄弟关系的精神相对待。
montgomery_911_calls: bundled 25,000-row sample of the 663,522-row original (the full file is on this machine but was not used, because prefer='sample') — every number below is for the sample, not the full file. How it was drawn: 1 row in every 27, evenly spread, so the sample covers the same 2015-12-10 to 2020-07-29 window with all 100 call types and 68 townships.

911 records: 25,000; columns: ['lat', 'lng', 'desc', 'zip', 'title', 'timeStamp', 'twp', 'addr', 'e']
Three descriptions, as logged:
  

## Task 1: Multi-Language Tokenization (30 points)

Implement one tokenizer per writing system and run all three on **Article 1** of their language:
1. **English** — word tokenization. `text.split()` is not enough: decide what happens to `rights.` and
   `(a)`. NLTK's `word_tokenize` or a regex are both acceptable.
2. **Arabic** — word tokenization *after* normalization. Strip diacritics (the file contains them —
   check), unify the alef forms (أ إ آ → ا), then split on runs of Arabic letters. Note in a comment what
   happens to `والحقوق` ("and-the-rights"): the conjunction and the article stay attached, and that is
   correct Arabic, not a bug in your code.
3. **Chinese** — character-level tokenization: Chinese has no spaces at all. Say in a comment what a real
   word segmenter (such as *jieba*) would change, and why character-level is the honest baseline when you
   do not have one.

Each function must return a list of strings. Print the tokens and the token count for Article 1 in each
language.


In [2]:
# Task 1 scaffold: three tokenizers, one per writing system. Different scripts break
# differently — English splits on spaces but glues punctuation on, Arabic attaches
# clitics to words and carries optional diacritics, Chinese has no spaces at all.

# NLTK is optional. If you use it, this line fetches its tokenizer model once (no-op afterwards).
# import nltk; nltk.download("punkt_tab", quiet=True); from nltk.tokenize import word_tokenize

ARABIC_DIACRITICS = re.compile(r"[ً-ْـ]")     # tashkeel marks and the tatweel stretch


def tokenize_english(text):
    # YOUR CODE HERE
    pass


def normalize_arabic(text):
    # YOUR CODE HERE: strip diacritics, unify alef forms
    pass


def tokenize_arabic(text):
    # YOUR CODE HERE: normalize first, then split on runs of Arabic letters (U+0621-U+064A)
    pass


def tokenize_chinese(text):
    # YOUR CODE HERE: one token per CJK character (U+4E00-U+9FFF); keep or drop punctuation — say which
    pass


# TODO: run each tokenizer on Article 1 of its language; print the tokens and the count.
# YOUR CODE HERE


## Task 2: Compare Tokenization Methods and Languages (25 points)

**(a) Three methods, one text.** On the full English text (all 28 articles joined), compare:
1. whitespace split — `text.split()`
2. your `tokenize_english` from Task 1
3. character 3-grams — `[text[i:i+3] for i in range(len(text) - 2)]`

For each: total tokens, distinct tokens, and the ten most frequent. Then write, in a comment, two concrete
places where methods 1 and 2 disagree (find them by set difference, do not guess).

**(b) Three languages, one content.** For every one of the 28 articles compute the token count in each
language with the Task 1 tokenizers (words for `en` and `ar`, characters for `zh`), then report:
- mean tokens per article per language, and the ratio `ar/en` and `zh/en`
- mean token length in characters per language
- type/token ratio (distinct ÷ total) per language

Say which language needs the most tokens for the *same* content, and connect it to the fertility numbers
Unit 2's example measured for BERT's tokenizer (English about 1.2 pieces per word, Arabic 5.5-6.1): if you
paid per token, which language would cost most, and why does *character* count not settle the question?


In [3]:
# TODO (Task 2a): compare split() / tokenize_english / character 3-grams on the English text.
# YOUR CODE HERE

# TODO (Task 2b): per-article token counts in the three languages, then the statistics above.
# YOUR CODE HERE


## Task 3: Real-World Application — Tokenizing a 911 Log (45 points)

The `desc` field was written by a dispatch system, not by a linguist. Look at the three printed above:
an address, a township, sometimes a station code, and a timestamp, separated by semicolons, in capitals,
with `&` for intersections and `@` before the time.

1. **Measure the naive approach.** Tokenize all 25,000 descriptions with `split()` and with your
   `tokenize_english`. For each: total tokens, distinct tokens, and the twenty most frequent tokens. What
   fraction of the distinct tokens are pure numbers?
2. **Respect the structure.** Write `tokenize_call(desc)` that splits on `;` first, then tokenizes each
   field, and returns a list of fields (each a token list). How many fields do descriptions have — always
   the same number? Print the distribution.
3. **Extract, do not just tokenize.** From the `title` column (`EMS: HEAD INJURY`), split off the category
   before the colon and count calls per category. From `desc`, pull the timestamp with a regex and check
   it against the `timeStamp` column: how many rows disagree?
4. **Write it down.** In a markdown cell, 3-5 sentences: what is a "token" for this source, which of
   your tokenizers you would ship, and one thing the data cell's honesty line about the sample means for
   the counts you just printed.

**Submission**: Complete notebook with all implementations and analysis.

**Grading**: 100 points total


In [4]:
# TODO (Task 3): tokenize the 911 descriptions three ways, measure each, then extract the
# category and the timestamp and check them against the structured columns.

# YOUR CODE HERE


## 💬 Discuss

1. Your Chinese tokenizer returns one character at a time, so `人权` ("human rights") becomes two tokens
   that a model must reassemble. Your Arabic tokenizer returns `والحقوق` as one token that a model must take
   apart. Which error is cheaper to live with in a search engine, and which in a classifier — and why does
   the answer differ?
2. The 911 log has a structure that a tokenizer can either respect (`;` first) or ignore. Name one
   downstream task where ignoring it is harmless and one where it silently corrupts the result.


## ⚠️ Where this breaks

**Character-level Chinese is a baseline, not a solution.** `人权` is one word in two characters; a bag of
characters cannot tell it from `权人`. Real systems run a word segmenter first, and the segmenter is itself
a model that can be wrong — an error you inherit before your own pipeline starts.

**Arabic clitics do not come off with a space split.** `والحقوق` carries "and" and "the" glued to
"rights". Unit 2's example measured the consequence inside a subword tokenizer: 5.5-6.1 pieces per Arabic
word against 1.2 for English on the same sentence. Every Arabic pipeline pays that cost or does
morphological analysis first.

**Corpora arrive truncated, and nobody tells you.** The English UDHR in NLTK's corpus stops in the middle
of Article 29; the Arabic and Chinese copies run to 30. A "same content" comparison run on the raw files
would have compared 28.5 articles with 30 and reported a false ratio. The data cell trimmed all three to
Articles 1-28 and says so. Check the ends of your files before you compare anything across them.

**The 911 sample is a sample.** The loader's first printed line says it: 25,000 of 663,522 rows, one in
every 27. Every count you print in Task 3 is a count on the sample; the *shape* holds (all call types and
townships survive), the *totals* do not.
